# Frozen distance adapter: eight equally shifted goals

Select **GPU → Run all**. No fitting or new-goal calibration occurs. Existing experiments/results remain unchanged.

Default: original **(50,42)** plus **eight goals**, displaced **sqrt32 ≈5.66units** at0,45,...315degrees. This covers horizontal, vertical and diagonal shifts, including the previous **(46,38)** test. Use the same final distance-adapter checkpoint, stage4/R34.

Every row uses **one common heldout set of ball positions across all nine goals**: inside the original fitting disk, within the trained distance range from every goal, and excluding fitting/reference coordinate/image overlaps.

The summary plot shows how transfer varies with shift direction. Each goal also gets a full-right heatmap. Calibration plots and reference images are saved; enable `SHOW_CALIBRATION` to display all scatters too.

Edit `SOURCE_RUN`, `SOURCE_STAGE` or `NEW_GOALS` below if desired. Small-stage/all-goal intersections may contain no heldout grid positions; these are N/A, not perfect scores.


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, shutil, signal, subprocess, sys, time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
DRIVE = Path('/content/drive/MyDrive')
SOURCE_RUN = DRIVE / 'datasets/curriculum_cost_adapter/radial_distance_v1/20261106T101715235909Z'
SOURCE_STAGE = 4
import math
GOAL_SHIFT = math.sqrt(32.0)
GOAL_ANGLES = list(range(0,360,45))
NEW_GOALS = [[round(50+GOAL_SHIFT*math.cos(math.radians(a)),6),
              round(42+GOAL_SHIFT*math.sin(math.radians(a)),6)] for a in GOAL_ANGLES]
SHOW_CALIBRATION = False
assert (SOURCE_RUN/'run.json').is_file(), 'Set SOURCE_RUN to your existing radial_distance_v1 run folder.'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = DRIVE / 'datasets/curriculum_cost_adapter/radial_goal_sweep_v1' / RUN_ID
REPO = Path('/content/i-jepa-wm-radial-goal-sweep')
REPO_URL = 'https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT = '8fc35399ea9578512f3016e6e6aad228298d2ad9'
DINO_ROOT = Path('/content/dino_wm')
DINO_COMMIT = '0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE = DRIVE / 'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
ENCODER = Path('/content/radial-goal-sweep-artifacts/encoder.pth.tar')
ENCODER_SHA = '3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'
assert ENCODER_DRIVE.is_file(), f'Missing encoder: {ENCODER_DRIVE}'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8*1024*1024), b''): h.update(block)
    return h.hexdigest()

def shell(command, cwd=None):
    command=list(map(str,command)); print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout: print(line,end='',flush=True)
            returncode=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.terminate()
            raise
    if returncode: raise subprocess.CalledProcessError(returncode,command)

print('GPU:', torch.cuda.get_device_name())
print('New Drive results:', OUTPUT)
print('Frozen source run:',SOURCE_RUN,'stage',SOURCE_STAGE)
print('New goals:',NEW_GOALS)


## Same saved adapter and frozen encoder

The source Drive run supplies the original distance-fitted adapter. Root/checkpoint hashes and the entire source lineage are verified. The old local feature cache is unnecessary. Historical fitting-code and current diagnostic fingerprints differ intentionally; backbone/render/precision checks and the original-goal control are retained.


In [ ]:
if not REPO.exists():
    shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
assert SOURCE_COMMIT, 'Use the published notebook with a verified source pin.'
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow'])
if not DINO_ROOT.exists(): shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ENCODER.parent.mkdir(parents=True,exist_ok=True)
if not ENCODER.exists() or sha256(ENCODER)!=ENCODER_SHA: shutil.copy2(ENCODER_DRIVE,ENCODER)
assert sha256(ENCODER)==ENCODER_SHA, 'Unexpected encoder weights.'
print('Ready. The saved distance-adapter run supplies the frozen weights; no training will run.')

def command(operation):
    if not NEW_GOALS: raise ValueError('NEW_GOALS must contain at least one goal.')
    args=[sys.executable,'-m','src.mylonas_ijepa_wm.radial_goal_transfer',operation,
          '--source-run',SOURCE_RUN,'--output-dir',OUTPUT,'--stage',SOURCE_STAGE]
    for goal in NEW_GOALS: args+=['--new-goal',*goal]
    if operation=='evaluate': args+=['--encoder-checkpoint',ENCODER,'--dino-wm-root',DINO_ROOT,'--device','cuda']
    return args


## Evaluate all goals with unchanged weights

The encoder sees no goal marker. Each new goal is represented by an image with the ball at that target position. Verify that changing the environment goal leaves fixed-position observations unchanged.

Encode every unique grid/reference coordinate once, batch8float32; references can span multiple batches. Reference tokens are reused for self-comparisons. Frozen weights and source checkpoint files must remain unchanged. Distance remains34×adaptedMSE; the raw scalar stays the original training calibration.

New timestamped outputs use `radial_goal_sweep_v1`. Rerunning this evaluation cell reuses validated results; changed goals/order/settings require a new setup timestamp.


In [ ]:
started=time.monotonic()
shell(command('evaluate'),cwd=REPO)
print(f'Frozen {len(NEW_GOALS)}-shift goal sweep took {(time.monotonic()-started)/60:.2f} minutes.')


## Compare goals on the same heldout ball positions

- Summary numbers match goal IDs in the table and heatmaps; goal0 is the original control.
- Mean absolute distance error (MAE) and RMSE use physical simulation units. Delta MAE subtracts the original control on the same ball positions.
- Uphill means cost increases while moving toward the evaluated goal along an adjacent grid edge. Equal-distance edges are omitted; step counts can differ by goal. Ties are shown separately.
- The primary heldout cohort is identical across all goals. Per-goal local/full-domain results are saved as secondary metrics; local populations differ and full-domain includes outside-region/range extrapolation.
- Original-control values may differ from the previous two-goal notebook because the all-goal intersection is stricter. Compare rows **within this sweep**.
- White points show original fitting positions; the dashed circle stays at the old fitting region. Black x is the original goal; orange star is the evaluated goal. Raw/error maps have separate scales; adapted/true maps share a full-range scale.
- Empty groups are N/A. Goal shifts are correlated tests of one checkpoint, not independent training seeds or planning trials.


In [ ]:
shell(command('report'),cwd=REPO)
r=json.loads((OUTPUT/'report.json').read_text())
from IPython.display import display,Image

def number(value): return 'N/A' if value is None else f'{value:.4f}'
def pct(value): return 'N/A' if value is None else f'{value:.1%}'

def delta(value,reference): return None if value is None or reference is None else value-reference

print('Frozen source stage/radius:',r['source']['stage'],r['source']['radius'])
print('Adapter unchanged:',r['adapter_unchanged'],'| scale:',r['distance_scale'],'| original raw scalar:',r['raw_calibration'])
print('Original-goal cached/fresh consistency:',r['original_goal_consistency'])
print('MAIN RESULTS: same heldout ball positions for every goal')
print(f"{'id':>3} {'goal':>20} {'shift':>7} {'N':>6} {'raw MAE':>9} {'adapt MAE':>10} {'RMSE':>9} {'delta MAE':>10} {'uphill':>8} {'steps':>7} {'ties':>6}")
control=r['goals'][0]['heldout_common']['adapted']['mae']
old=r['goals'][0]['goal']
for j,g in enumerate(r['goals']):
    m=g['heldout_common'];e=g['heldout_directional']
    shift=math.hypot(g['goal'][0]-old[0],g['goal'][1]-old[1])
    name=str(tuple(round(v,3) for v in g['goal']))
    print(f"{j:>3} {name:>20} {shift:>7.3f} {m['count']:>6} {number(m['raw_calibrated']['mae']):>9} "
          f"{number(m['adapted']['mae']):>10} {number(m['adapted']['rmse']):>9} "
          f"{number(delta(m['adapted']['mae'],control)):>10} {pct(e['adapted_up_fraction']):>8} {e['comparisons']:>7} {e['adapted_ties']:>6}")
print('The shared cohort is inside the original fitting disk and within the fitted distance range for ALL goals.')
display(Image(filename=str(OUTPUT/'goal_summary.png')))
for j,g in enumerate(r['goals']):
    print('\nGOAL',j,g['goal'],'| raw/adapted self-cost:',g['raw_self_cost'],g['adapted_self_cost'])
    print('Heldout raw/adapted uphill:',pct(g['heldout_directional']['raw_up_fraction']),pct(g['heldout_directional']['adapted_up_fraction']))
    display(Image(filename=str(OUTPUT/f'goal_{j}_heatmaps.png')))
    if SHOW_CALIBRATION:display(Image(filename=str(OUTPUT/f'goal_{j}_calibration.png')))


## Summary across goal shifts

Lower distance errors/uphill fractions are better. Read the original-goal control first. Exact zero self-cost is expected from comparing a reference with itself; useful transfer is measured away from the reference. One shift outperforming another is descriptive evidence of directional dependence, not a significance test.


In [ ]:
tested=[(j,g) for j,g in enumerate(r['goals']) if j and g['heldout_common']['adapted']['mae'] is not None]
if not tested:
    print('No heldout common positions for this goal set/checkpoint. Results are N/A; do not infer zero error.')
else:
    best=min(tested,key=lambda item:item[1]['heldout_common']['adapted']['mae'])
    worst=max(tested,key=lambda item:item[1]['heldout_common']['adapted']['mae'])
    better=sum(g['heldout_common']['adapted']['mae']<g['heldout_common']['raw_calibrated']['mae'] for _,g in tested)
    print('Original-goal heldout MAE:',number(control))
    print('Lowest changed-goal MAE:',best[0],best[1]['goal'],number(best[1]['heldout_common']['adapted']['mae']))
    print('Highest changed-goal MAE:',worst[0],worst[1]['goal'],number(worst[1]['heldout_common']['adapted']['mae']))
    print(f'Adapter improves on original-calibration raw baseline for {better}/{len(tested)} tested shifts.')
print('No optimization or goal-specific calibration occurred. All numerical reports/reference images/calibration plots are saved.')
print('Full-right maps include outside-region points; headline metrics exclude them.')
print('Results:',OUTPUT)


## Return the executed notebook

Download **File → Download → Download .ipynb** and attach it here. The source adapter experiment is unchanged. Every goal reference, score array, heatmap and calibration plot is retained in the new output folder, alongside the goal-summary figure and provenance.

Local synthetic checks verify the workflow. Actual GPU results establish transfer for these shifts; they do not establish arbitrary-goal or planning performance.
